In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("../lessons/WineQT.csv")

X = df.drop(["quality", "Id"], axis=1).values
y = (df["quality"] > 5).astype(int).values

# делим на train/test руками (без sklearn): перемешиваем и режем 80/20
rng = np.random.default_rng(42)          # фиксируем случайность, чтобы результат повторялся
idx = rng.permutation(len(X))            # перемешанные номера строк
n_train = int(len(X) * 0.8)
train_idx, test_idx = idx[:n_train], idx[n_train:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# нормализация руками: (значение - среднее) / разброс.
# ВАЖНО: среднее и std считаем ТОЛЬКО по train, применяем к обоим.
mean = X_train.mean(axis=0)
std  = X_train.std(axis=0)

X_train = (X_train - mean) / std
X_test  = (X_test - mean) / std

In [ ]:
# --- КЛАСС SVM ---
class SVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = learning_rate          # размер шага градиентного спуска
        self.lambda_param = lambda_param # сила "растягивания" полосы (регуляризация)
        self.n_iters = n_iters           # сколько раз пройти по всем данным
        self.w = None                    # веса (по одному на признак)
        self.b = None                    # смещение

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # метки переводим из {0, 1} в {-1, +1} — так устроена формула SVM
        y_ = np.where(y <= 0, -1, 1)

        # стартуем с нулей
        self.w = np.zeros(n_features)
        self.b = 0.0

        # градиентный спуск: n_iters раз проходим по всем точкам
        for _ in range(self.n_iters):
            for i in range(n_samples):
                xi = X[i]
                # margin = y * (w·x + b): насколько уверенно и правильно классифицировано
                margin = y_[i] * (np.dot(xi, self.w) + self.b)

                if margin >= 1:
                    # точка правильная и далеко от полосы -> только расширяем полосу
                    dw = self.lambda_param * self.w
                    db = 0
                else:
                    # точка ошибочная/близко -> ещё и толкаем к правильному ответу
                    dw = self.lambda_param * self.w - y_[i] * xi
                    db = -y_[i]

                # шаг: двигаем веса и смещение против градиента (в сторону меньшего штрафа)
                self.w -= self.lr * dw
                self.b -= self.lr * db

    def decision_function(self, X):
        # "сырое" число w·x + b для каждой строки
        return np.dot(X, self.w) + self.b

    def predict(self, X):
        # знак числа -> класс. Возвращаем 0/1, чтобы сравнивать с исходными метками.
        return np.where(self.decision_function(X) >= 0, 1, 0)

In [ ]:
# --- Обучаем и проверяем ---
model = SVM(learning_rate=0.001, lambda_param=0.01, n_iters=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

# accuracy руками: доля совпавших ответов
accuracy = (predictions == y_test).mean()
print("Наш SVM, accuracy на тесте:", round(accuracy, 3))

In [ ]:
# --- Сверимся с sklearn (LinearSVC) на тех же данных ---
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

sk = LinearSVC(max_iter=10000)
sk.fit(X_train, y_train)
sk_pred = sk.predict(X_test)

print("sklearn LinearSVC, accuracy:", round(accuracy_score(y_test, sk_pred), 3))
print("Наш SVM,           accuracy:", round(accuracy, 3))